In [1]:
!pip install transformers datasets accelerate torch


In [2]:
import os
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    pipeline,
)

# =========================
# 1. Configuration
# =========================

MODEL_NAME = "microsoft/DialoGPT-medium"
DATASET_PATH = "/content/egypt_tourism_gpt2_plain_text.txt"
OUTPUT_DIR = "/content/egypt-tourism-gpt2-model"

BLOCK_SIZE = 256

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


# =========================
# 2. Load tokenizer and model
# =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# DialoGPT/GPT-2 has no default padding token.
# We use EOS as PAD, but we will handle labels manually.
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))


# =========================
# 3. Load dataset correctly
# =========================

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Dataset file not found: {DATASET_PATH}")

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

examples = raw_text.split("<|endoftext|>")

examples = [
    example.strip() + "\n<|endoftext|>"
    for example in examples
    if example.strip()
]

print("Number of conversation examples:", len(examples))

dataset = Dataset.from_dict({"text": examples})

dataset = dataset.train_test_split(
    test_size=0.1,
    seed=42
)

print(dataset)


# =========================
# 4. Tokenization with manual labels
# =========================

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=BLOCK_SIZE,
        padding="max_length"
    )

    labels = []

    for input_ids, attention_mask in zip(
        tokenized["input_ids"],
        tokenized["attention_mask"]
    ):
        label = []

        for token_id, mask in zip(input_ids, attention_mask):
            if mask == 1:
                label.append(token_id)
            else:
                label.append(-100)

        labels.append(label)

    tokenized["labels"] = labels

    return tokenized


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print(tokenized_dataset)


# =========================
# 5. Training arguments
# =========================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    learning_rate=2e-5,
    weight_decay=0.01,

    logging_steps=10,

    eval_strategy="steps",
    eval_steps=50,

    save_steps=50,
    save_total_limit=2,

    report_to="none",

    fp16=False,
    bf16=False
)


# =========================
# 6. Trainer
# =========================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
)


# =========================
# 7. Start training
# =========================

trainer.train()


# =========================
# 8. Save model
# =========================

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Model saved to: {OUTPUT_DIR}")


# =========================
# 9. Test generation safely on CPU first
# =========================

generator = pipeline(
    "text-generation",
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    device=-1
)

prompt = """<|system|>
You are an Egypt tourism guide assistant. Answer clearly, politely, and provide practical tourism advice.
<|user|>
Tell me about the Pyramids of Giza.
<|assistant|>
"""

result = generator(
    prompt,
    max_new_tokens=100,
    do_sample=False,
    repetition_penalty=1.15,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id
)

print("\nGenerated Answer:\n")
print(result[0]["generated_text"])

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of conversation examples: 120
DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 108
    })
    test: Dataset({
        features: ['text'],
        num_rows: 12
    })
})


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/egypt-tourism-gpt2-model


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens', 'eos_token_id', 'do_sample', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Answer:

<|system|>
You are an Egypt tourism guide assistant. Answer clearly, politely, and provide practical tourism advice.
<|user|>
Tell me about the Pyramids of Giza.
<|assistant|>
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


In [15]:
OUTPUT_DIR = "/content/egypt-tourism-gpt2-model"
prompt = """<|system|>
You are an Egypt tourism guide assistant. Answer clearly, politely, and provide practical tourism advice.
<|user|>
Suggest a one-day trip in Giza.
<|assistant|>
"""
result = generator(
    prompt,
    max_new_tokens=100,
    do_sample=False,
    repetition_penalty=1.15,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id
)

print("\nGenerated Answer:\n")
print(result[0]["generated_text"])

Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Answer:

<|system|>
You are an Egypt tourism guide assistant. Answer clearly, politely, and provide practical tourism advice.
<|user|>
Suggest a one-day trip in Giza.
<|assistant|>
For more information about Egyptian pyramids visit: http://www2.googleapis.com/en_US or call 1-800–829‐9000 for assistance.


In [3]:
!rm -rf /content/egypt-tourism-gpt2-model

In [17]:
!pip install -U transformers datasets accelerate peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 72.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [4]:
import os
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    pipeline,
)
from peft import LoraConfig
from trl import SFTTrainer

# =========================
# 1. Configuration
# =========================

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DATASET_PATH = "/content/egypt_tourism_gpt2_plain_text.txt"
OUTPUT_DIR = "/content/egypt-tourism-qwen-lora"

MAX_SEQ_LENGTH = 512

print("CUDA available:", torch.cuda.is_available())


# =========================
# 2. Load dataset
# =========================

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Dataset file not found: {DATASET_PATH}")

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

examples = raw_text.split("<|endoftext|>")

examples = [
    example.strip()
    for example in examples
    if example.strip()
]

print("Number of examples:", len(examples))

dataset = Dataset.from_dict({"text": examples})

# Use the whole dataset for training because the dataset is very small.
train_dataset = dataset


# =========================
# 3. Load tokenizer and model
# =========================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False


# =========================
# 4. LoRA configuration
# =========================

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)


# =========================
# 5. Training arguments
# =========================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=15,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    weight_decay=0.01,

    logging_steps=5,
    save_steps=50,
    save_total_limit=2,

    fp16=torch.cuda.is_available(),
    bf16=False,

    report_to="none",
    remove_unused_columns=False,

    optim="adamw_torch"
)


# =========================
# Upgrade torchao to be compatible with peft
# =========================
!pip install --upgrade torchao


# =========================
# 6. SFT Trainer
# =========================

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
)

trainer.train()


# =========================
# 7. Save model adapter and tokenizer
# =========================

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved to:", OUTPUT_DIR)

CUDA available: True
Number of examples: 120


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 108.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Adding EOS to train dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,2.719913
10,1.325490
15,1.065800
20,0.868299
25,0.619672
30,0.585117
35,0.419471
40,0.310331
45,0.407434
50,0.312743


Model saved to: /content/egypt-tourism-qwen-lora


In [6]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch

OUTPUT_DIR = "/content/egypt-tourism-qwen-lora"

tokenizer = AutoTokenizer.from_pretrained(
    OUTPUT_DIR,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

messages = [
    {
        "role": "system",
        "content": "You are an Egypt tourism guide assistant. Answer clearly, politely, and provide practical tourism advice. Do not invent facts."
    },
    {
        "role": "user",
        "content": "Tell me about Pyramids of Giza."
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

generated = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(generated.strip())

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/336 [00:00<?, ?it/s]

The Pyramids of Giza are among Egypt's most famous ancient monuments. They include the pyramids of Khufu, Khafre, and Menkaure, and are located near the Great Sphinx. Tourist tip: Visit early in the morning or near sunset, wear comfortable shoes, bring water, and consider hiring a licensed guide.


In [10]:
import shutil
import os

# Define the output directory (where the model is saved)
OUTPUT_DIR = "/content/egypt-tourism-gpt2-model"

# Define the path for the zip file
zip_file_path = "/content/egypt-tourism-gpt2-model.zip"

# Create a zip archive of the model directory
# The base_name is the path to the output zip file (without .zip extension)
# The format is 'zip'
# The root_dir is the directory to start zipping from
# The base_dir is the directory inside root_dir to archive
shutil.make_archive(
    base_name=zip_file_path.replace('.zip', ''),
    format='zip',
    root_dir=os.path.dirname(OUTPUT_DIR),
    base_dir=os.path.basename(OUTPUT_DIR)
)

print(f"Model archived to: {zip_file_path}")
print("You can now download this file from the Colab file browser.")

Model archived to: /content/egypt-tourism-gpt2-model.zip
You can now download this file from the Colab file browser.


In [11]:
from google.colab import drive
import shutil
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the source folder (the model output directory)
SOURCE_DIR = "/content/egypt-tourism-qwen-lora"

# Define the destination path in Google Drive
# This will create a folder named 'qwen_lora_model' in your Drive's root
DEST_DIR = "/content/drive/My Drive/qwen_lora_model"

print(f"Copying {SOURCE_DIR} to {DEST_DIR}...")

# Check if the source directory exists
if not os.path.exists(SOURCE_DIR):
    print(f"Error: Source directory {SOURCE_DIR} does not exist.")
else:
    # Create the destination directory if it doesn't exist
    os.makedirs(DEST_DIR, exist_ok=True)

    # Copy the contents of the source directory to the destination
    # Using 'copytree' to copy the folder and its contents
    shutil.copytree(SOURCE_DIR, DEST_DIR, dirs_exist_ok=True)
    print(f"Successfully copied {SOURCE_DIR} to {DEST_DIR}.")
    print("You can now find your model in your Google Drive.")

Mounted at /content/drive
Copying /content/egypt-tourism-qwen-lora to /content/drive/My Drive/qwen_lora_model...
Successfully copied /content/egypt-tourism-qwen-lora to /content/drive/My Drive/qwen_lora_model.
You can now find your model in your Google Drive.
